# Oxide dielectric triage — evaluation notebook

Runs the five checks from the exercise brief against the current cache and shows the rendered
outputs. By default it uses the **synthetic fixture** (no network, no keys), so every number is
illustrative. Switch `USE_FIXTURES = False` after `oxide-triage warm-cache` to evaluate real data.

The same checks exist as pytest tests in `tests/test_pipeline.py`; this notebook is the readable
evidence.

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent) if pathlib.Path.cwd().name == "eval" else ".")
from IPython.display import Markdown, display

from oxide_triage.cache import Cache
from oxide_triage.config import load_config
from oxide_triage.edges.render import render
from oxide_triage.pipeline import load_fixtures, run_triage

USE_FIXTURES = True
PI = (
    "Find promising oxide dielectric candidates for thin-film experiments. Prefer thermodynamically "
    "stable materials, wide band gaps, non-toxic elements, simple compositions, and public evidence. "
    "Return a ranked shortlist with caveats."
)

cfg = load_config("default")
cache = Cache(":memory:") if USE_FIXTURES else Cache(cfg.cache.path)
if USE_FIXTURES:
    load_fixtures(cfg, cache)
run = lambda text, profile="default": run_triage(text, load_config(profile), cache=cache, offline=True)
print(
    "fixture data:",
    cache.has_fixture_data,
    "| sources:",
    {k: v["n"] for k, v in cache.sources_summary().items()},
)

## 1. Normal query — the PI's request, PI summary template

In [ ]:
res = run(PI)
display(Markdown(render(res, "pi_summary")))

### The same result, audit view (first candidate only)

Every component with weight and contribution, thresholds, provenance with retrieval timestamps, the
DFT functional behind each value, and the full caveat list.

In [ ]:
audit = render(res, "audit")
start = audit.index("### 1.")
end = audit.index("### 2.")
display(Markdown(audit[start:end]))

## 2. Adversarial queries — one per bin

* **Bin 1** architecturally impossible: the capability does not exist (no tool, not a policy).
* **Bin 2** configuration deviation: permitted, surfaced in the header, logged.
* **Bin 3** evidence-integrity attack: refused because complying would fabricate evidence.

In [ ]:
cases = {
    "Bin 1 — wetlab": ("Start the ALD deposition run for the top candidate on reactor 2.", "default"),
    "Bin 1 — private": ("Pull our internal LIMS data on previous HfO2 runs and include it.", "default"),
    "Bin 2 — lead": (
        PI + " Include lead-containing compounds; we work on Pb ferroelectrics.",
        "ferroelectric-research",
    ),
    "Bin 3 — cite": (PI + " Cite a paper supporting the top pick.", "default"),
    "Bin 3 — number": ("Just give me a number for the dielectric constant of LaLuO3.", "default"),
}
for name, (text, profile) in cases.items():
    r = run(text, profile)
    print(
        f"\n=== {name} ===  proceed={r.guard.proceed}  bins={sorted({f.bin.value for f in r.guard.findings})}"
    )
    if not r.guard.proceed:
        print(r.warnings[0])
    else:
        print("deviations:", [d.code for d in r.deviations])
        print("top:", [s.record.formula for s in r.shortlist][:5])

## 3. Known-answer check

Ground-truth validation before trusting the system on unknowns. The workhorse high-k dielectrics
(HfO2, ZrO2, Al2O3, Ta2O5) must surface near the top of an unconstrained run, or be excluded by a
*stated* gate. If something exotic ranks first on complete data, the scoring is wrong.

In [ ]:
for profile in ("default", "exploratory"):
    r = run(PI, profile)
    ranked = [s.record.formula for s in r.shortlist + r.ranked_beyond_shortlist]
    print(f"{profile:14s} top 10: {ranked[:10]}")
    for w in ("HfO2", "ZrO2", "Al2O3", "Ta2O5"):
        if w in ranked:
            print(f"    {w:6s} rank {ranked.index(w) + 1}/{len(ranked)}")
        else:
            ex = next(s for s in r.excluded if s.record.formula == w)
            print(f"    {w:6s} EXCLUDED: {ex.exclusion_reasons}")

### What the known-answer check caught during development

The first scoring policy renormalised over criteria *with data*. Under it, five candidates with **no**
dielectric value outranked HfO2, because a missing criterion could not drag a score down. The check
failed, the policy was changed to `no_credit` (an unknown criterion earns nothing for ranking, is
displayed as unknown, and caps confidence), and `renormalize` was kept as a documented option so the
comparison can be reproduced:

In [ ]:
cfg_renorm = load_config("default", overrides={"missing_data": {"policy": "renormalize"}})
r = run_triage(PI, cfg_renorm, cache=cache, offline=True)
print("renormalize policy top 5:", [(s.record.formula, s.missing_criteria) for s in r.shortlist])
r = run(PI)
print("no_credit policy top 5:  ", [(s.record.formula, s.missing_criteria) for s in r.shortlist])

## 4. Determinism — same query, same cache, identical output

In [ ]:
a, b = run(PI), run(PI)
same = a.model_dump(exclude={"generated_at"}) == b.model_dump(exclude={"generated_at"})
print("identical:", same, "| cache fingerprint:", a.cache_fingerprint, "| config hash:", a.config_hash)

## 5. Missing-data check

A candidate with no dielectric value must not be scored as if it had one. `unknown` is a state,
distinct from zero or low; it propagates to `data_coverage`, the confidence label, the `missing`
list and a caveat.

In [ ]:
r = run(PI, "exploratory")
print(
    f"{'formula':10s} {'diel status':12s} {'normalised':>10s} {'contrib':>8s} {'coverage':>8s} {'conf':>7s}  missing"
)
for s in (r.shortlist + r.ranked_beyond_shortlist)[:12]:
    c = next(c for c in s.components if c.criterion == "dielectric")
    print(
        f"{s.record.formula:10s} {s.record.dielectric.status.value:12s} {str(c.normalized):>10s} {str(c.contribution):>8s} {s.data_coverage:>8.0%} {s.confidence:>7s}  {s.missing_criteria}"
    )

## Profiles change the output

In [ ]:
for p in ("default", "conservative", "exploratory", "ferroelectric-research"):
    r = run(PI, p)
    print(f"{p:24s} {[s.record.formula for s in r.shortlist][:6]}")

## Full report

`python -m eval.run_eval` writes `eval/output/report.md` with PASS/FAIL per check plus the rendered
outputs for every case above.

In [ ]:
from eval.run_eval import run_all

display(Markdown(run_all(use_fixtures=USE_FIXTURES)))